In [ ]:
# imports
import pandas as pd
from datasets import load_dataset
from evaluator import Evaluator

In [ ]:
# load the Splits! dataset
splits = load_dataset("ecaplan/splits", "splits")
# all versions of the dataset contain only a 'train' portion
splits = splits['train']

In [ ]:
# convert the huggingface dataset to a pandas dataframe
df = pd.DataFrame(splits)

In [ ]:
# since the full dataset has >176k samples, we will use a smaller sample for testing
sample_size = 10
# sample the dataframe
sample_df = df.sample(sample_size, random_state=42)
sample_df

In [ ]:
# now let's make some theories!
# in this example, we will make a really simple theory that says that group A uses more commas than group B when talking about the topic
# in general, your method can use the two groups, the topic, and the calibration set to make a theory
theories = []

for i, row in sample_df.iterrows():
    demo_A = row['demographic_A']
    demo_B = row['demographic_B']
    topic = row['description']
    calibration_set = row['example_posts']

    # make a theory for each row
    theory = f"People who are '{row['demographic_A']}' use more commas than people who are '{row['demographic_B']}' when talking about '{row['description']}'."
    theories.append(theory)



sample_df['theory'] = theories
sample_df

In [ ]:
# if quantize=True, it will load the Llama-3.1-70B model in 8-bit
# for our experiments, we used full precision, so we recommend using quantize=False if you have enough memory
quantize = True
# batch size for ~4k tokens (raise at your own risk)
batch_size = 1

# now let's load the classification model (make sure you have access to meta-llama/Llama-3.1-70B-Instruct through huggingface)
evaluator = Evaluator(sample_df, 
                      quantize=quantize,
                      batch_size=batch_size)

In [ ]:
# and let's evaluate the theories! (this can take a while)
evaluator.evaluate()

In [ ]:
# now we can see the results according to many splits

# pure accuracy
accuracy = evaluator.get_accuracy()
print(f"Accuracy: {accuracy}")

In [ ]:
# split by pairs of demographics
evaluator.split_by_demographic_A_demographic_B()

In [ ]:
# by a individual demographic
evaluator.split_by_unique_demographic()

In [ ]:
# by topic category
evaluator.split_by_category()

In [ ]:
# by demographic pair and topic category
evaluator.split_by_category_demographic_A_demographic_B()

In [ ]:
# by individual demographic and topic category
evaluator.split_by_category_unique_demographic()

In [ ]:
# by topic and individual demographic
evaluator.split_by_description_unique_demographic()

In [ ]:
# by topic and demographic pair
evaluator.split_by_description_demographic_A_demographic_B()

In [ ]:
# to get all splits, you can use the following function
all_splits = evaluator.get_all_split_reports()
all_splits